# 2. Compare performance and availability

This notebook builds a quick box-score score for the 2024–25 season and adjusts it for games played. The weights are simple choices for exploration, not an official NBA impact metric.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data") / "raw" / "statistics" / "player_stats_2024.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} player rows from {DATA_PATH}")
df.head()

## Score production and availability

The first score combines points, rebounds, assists, steals, and blocks. Availability is games played divided by 82, the full regular-season schedule.

In [ ]:
df["performance_score"] = (
    df["PTS"]
    + 1.2 * df["REB"]
    + 1.5 * df["AST"]
    + 3 * df["STL"]
    + 3 * df["BLK"]
)
df["availability"] = df["GP"] / 82
df["risk_adj_performance"] = df["performance_score"] * df["availability"]

df[["PLAYER_NAME", "TEAM_ABBREVIATION", "performance_score", "availability", "risk_adj_performance"]].sort_values(
    "risk_adj_performance", ascending=False
).head(20)

## Add a turnover penalty

This version subtracts 1.5 points per turnover. Comparing both versions shows how that choice changes the order.

In [ ]:
df["performance_score_v2"] = (
    df["PTS"]
    + 1.2 * df["REB"]
    + 1.5 * df["AST"]
    + 3 * df["STL"]
    + 3 * df["BLK"]
    - 1.5 * df["TOV"]
)
df["risk_adj_v2"] = df["performance_score_v2"] * df["availability"]

df[["PLAYER_NAME", "risk_adj_v2"]].sort_values(
    "risk_adj_v2", ascending=False
).head(20)